In [1]:
import dgl

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

In [11]:
cpg = pd.read_pickle("/root/autodl-tmp/vul-detect/data/cpg/0_cpg.pkl")

In [5]:
data = pd.read_pickle("/root/autodl-tmp/vul-detect/utils/data/torch_geometrics_process/cfexplainer/storage/processed/vul_graph_dataset/None_processed/devign_dataframe.pkl")

In [29]:
type(data.loc[123, "torch_geometrics_data"][0])

torch_geometric.data.data.Data

In [102]:
idx = 7826
l1 = data.loc[idx, "torch_geometrics_data"][0].edge_index
l2 = data.loc[idx, "torch_geometrics_data"][1].edge_index
l3 = data.loc[idx, "torch_geometrics_data"][2].edge_index
l = l1[0].tolist() + l1[1].tolist() + l2[0].tolist() + l2[1].tolist() + l3[0].tolist() + l3[1].tolist() 
print(len(set(l)))
data.loc[idx, "torch_geometrics_data"][0].x.shape[0]

19


20

In [4]:
# 定义存储不符合规则的行索引列表
invalid_rows = []

sum = 0
# 遍历每一行数据
for index, row in data.iterrows():
    # 获取 edge_index 数据
    torch_geometrics_data = row["torch_geometrics_data"]
    l1, l2, l3 = torch_geometrics_data[0].edge_index, torch_geometrics_data[1].edge_index, torch_geometrics_data[2].edge_index
    
    # 构建 l 列表
    l = (
        l1[0].tolist() + l1[1].tolist() +
        l2[0].tolist() + l2[1].tolist() +
        l3[0].tolist() + l3[1].tolist()
    )
    sum += len(set(l))
    # 检查规则
    if len(set(l)) != torch_geometrics_data[0].x.shape[0]:
        invalid_rows.append(index)

print(sum / data.shape[0])
# 输出不符合规则的行索引
print("不符合规则的行索引：", invalid_rows)

NameError: name 'data' is not defined

In [6]:
dgl = torch.load("/root/autodl-tmp/vul-detect/utils/data/torch_geometrics_process/cfexplainer/storage/processed/vul_graph_dataset/None_processed/data.pt")

In [7]:
len(dgl)

15584

In [5]:
import dgl.data
from dgl.dataloading import GraphDataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from dgl.data import DGLDataset

class vulDGLDataset(DGLDataset):
    def __init__(self, name, raw_dataframe_path=None, url=None, raw_dir=None, save_dir=None, hash_key=..., force_reload=False, verbose=False, transform=None):
        
        self.raw_dataframe_path = raw_dataframe_path
        super().__init__(name, url, raw_dir, save_dir, hash_key, force_reload, verbose, transform)
        self.data_list = torch.load(self.save_dir)
        # self.df = pd.read_pickle(raw_dataframe_path)
    def process(self):
        self.df = pd.read_pickle(self.raw_dataframe_path)
        self.label = self.df["target"].tolist()
        
    def __getitem__(self, idx):
        return idx, self.data_list[idx], self.label[idx]
    def __len__(self):
        return len(self.data_list)
        
raw_dataframe_path = "/root/autodl-tmp/vul-detect/utils/data/torch_geometrics_process/cfexplainer/storage/processed/vul_graph_dataset/None_processed/devign_dataframe.pkl"
dataset = vulDGLDataset("devign", raw_dataframe_path=raw_dataframe_path, save_dir="/root/autodl-tmp/vul-detect/utils/data/torch_geometrics_process/cfexplainer/storage/processed/vul_graph_dataset/None_processed/data.pt")

In [30]:
dataset[1]

(1,
 Graph(num_nodes={'node': 26},
       num_edges={('node', 'ast', 'node'): 45, ('node', 'cfgcdg', 'node'): 53, ('node', 'pdg', 'node'): 98},
       metagraph=[('node', 'node', 'ast'), ('node', 'node', 'cfgcdg'), ('node', 'node', 'pdg')]),
 1)

In [33]:
len(dataset)

15584

In [7]:
num_examples = len(dataset)
num_train = int(num_examples * 0.8)

train_sampler = SubsetRandomSampler(torch.arange(num_train))
train_dataloader = GraphDataLoader(dataset, sampler=train_sampler, batch_size=5, drop_last=False)

In [39]:
import math
import xgboost as xgb

def process_dump(dump_result):
    nodes = []
    for tree in dump_result:
        stack = []
        lines = tree.split('\n')
        for line in lines:
            if line.startswith('booster'):
                continue
            parts = line.strip().split(':')
            if len(parts) < 2:
                continue
            node_info = parts[0].strip()
            condition = parts[1].strip()
            if 'leaf' in node_info:
                logit = float(node_info.split('=')[1].strip())
                conversion_rate = 1 / (1 + math.exp(-logit))
                path = []
                while stack:
                    path.append(stack.pop())
                path.reverse()
                node_depth = len(path)
                nodes.append((conversion_rate, '\t' * node_depth + node_info +':'+ '->'.join(path)))
            else:
                stack.append(condition)
    nodes.sort(reverse=True)
    for conversion_rate, node_info in nodes:
        print(f"{conversion_rate:.4f}: {node_info}")

tensor([12360,  6003,  6526, 10998,  4969])
Graph(num_nodes={'node': 86},
      num_edges={('node', 'ast', 'node'): 139, ('node', 'cfgcdg', 'node'): 210, ('node', 'pdg', 'node'): 281},
      metagraph=[('node', 'node', 'ast'), ('node', 'node', 'cfgcdg'), ('node', 'node', 'pdg')])
[('node', 'ast', 'node'), ('node', 'cfgcdg', 'node'), ('node', 'pdg', 'node')]
torch.Size([86, 3, 10, 768])
tensor([0, 1, 1, 0, 1])
True
tensor([[1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        [1., 0., 1.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 1., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])
POS: tensor(indices=tensor([[ 0,  0,  1,  1,  2,  3,  4,  5,  6,  7,  7,  8,  9, 10,
                        11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24,
                        25, 26, 27, 28, 28, 29, 29, 30, 31, 32, 33, 34, 35, 36,
                        37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 46, 47, 47, 48,
   

In [37]:
import torch.nn as nn
layer = nn.Linear(10, 5, bias=True)
layer.weight

Parameter containing:
tensor([[ 0.1601,  0.0201, -0.0848,  0.0039, -0.1807, -0.0630, -0.2280, -0.0790,
         -0.2523,  0.1964],
        [-0.1132,  0.2944, -0.1373,  0.1872, -0.0517,  0.0768,  0.1562,  0.0839,
         -0.1296,  0.2310],
        [ 0.2323, -0.2929, -0.0583, -0.0132,  0.3046, -0.0547,  0.1405,  0.2136,
         -0.1094, -0.2062],
        [ 0.0895, -0.2131,  0.1257, -0.0021,  0.2923,  0.2675, -0.0477,  0.0704,
         -0.0741, -0.0218],
        [-0.1754, -0.1311, -0.0866,  0.1110,  0.0080,  0.1814,  0.2150,  0.1980,
          0.2491, -0.0996]], requires_grad=True)